# Used Cars Price Prediction — ML Walkthrough
**Person 2: ML Engineer** | MSiA 423 Cloud Engineering Team Project

This notebook walks through the full ML lifecycle end-to-end:

| Section | What it does |
|---|---|
| 1. Setup | Load config, connect to S3 |
| 2. Load Data | Read cleaned Parquet from S3 (or local) |
| 3. EDA | Distributions, missing values, correlations |
| 4. Feature Engineering | Build the sklearn preprocessing pipeline step-by-step |
| 5. Linear Baseline | Ridge regression — train, cross-validate, evaluate |
| 6. Tree Baseline | Random Forest — train, cross-validate, evaluate |
| 7. Challenger Model | XGBoost with early stopping |
| 8. Model Comparison | Side-by-side metrics, feature importance |
| 9. Save Artifacts | `best_model.pkl`, manifest, metrics → S3 or local |

> **Run locally or on EC2.** For S3 access, set `USE_S3 = True` below and make sure your AWS credentials / IAM role are configured. For a quick local dry-run, set `USE_S3 = False` and point `LOCAL_PARQUET` at a downloaded copy of the cleaned data.

---
## 1. Setup

In [ ]:
# ── Install deps in the *active notebook kernel* (run once if Parquet fails) ─
import sys
!{sys.executable} -m pip install -q pyarrow scikit-learn xgboost pyyaml matplotlib seaborn boto3

In [ ]:
import io
import json
import logging
import os
import pickle
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# Matplotlib style
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

print("✓ Imports OK")

In [ ]:
# ── Toggle S3 vs local mode ─────────────────────────────────────────────────
USE_S3 = False         # set True on EC2 with IAM role
# Paths are resolved from repo root (folder with config.yaml), not notebook cwd
LOCAL_PARQUET = "out/vehicles_clean.parquet"
LOCAL_ARTIFACTS = "out/models/latest"

# ── Load config.yaml ────────────────────────────────────────────────────────
def _repo_root() -> Path:
    for base in (Path.cwd().resolve(), Path.cwd().resolve() / "notebooks"):
        for candidate in (base, base.parent):
            if (candidate / "config.yaml").is_file():
                return candidate
    return Path.cwd().resolve()

REPO_ROOT = _repo_root()
CONFIG_PATH = REPO_ROOT / "config.yaml"
with open(CONFIG_PATH) as f:
    CFG = yaml.safe_load(f)

# Allow env-var override of bucket name (IAM role provides credentials)
if os.environ.get("USED_CARS_BUCKET"):
    CFG["aws"]["bucket"] = os.environ["USED_CARS_BUCKET"]

BUCKET = CFG["aws"]["bucket"]
REGION = CFG["aws"]["region"]
PROCESSED_KEY = CFG["aws"]["processed_key"]

print(f"Bucket : {BUCKET}")
print(f"Region : {REGION}")
print(f"Data   : s3://{BUCKET}/{PROCESSED_KEY}")

---
## 2. Load Data

In [ ]:
# Run the "Section 1 — Setup" cell above first (defines USE_S3, CFG, paths).
if "USE_S3" not in globals():
    raise RuntimeError(
        "Run Section 1 setup cells first (USE_S3, LOCAL_PARQUET, config.yaml)."
    )


def ensure_pyarrow_available() -> None:
    """Make pyarrow visible to this notebook kernel for Parquet reads."""
    import importlib
    import site
    import subprocess

    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)

    try:
        importlib.import_module("pyarrow")
        return
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "pyarrow"])
        importlib.invalidate_caches()
        if user_site not in sys.path:
            sys.path.append(user_site)
        importlib.import_module("pyarrow")


ensure_pyarrow_available()

if USE_S3:
    import boto3
    s3 = boto3.client("s3", region_name=REGION)
    logger.info("Downloading s3://%s/%s ...", BUCKET, PROCESSED_KEY)
    obj = s3.get_object(Bucket=BUCKET, Key=PROCESSED_KEY)
    df_raw = pd.read_parquet(io.BytesIO(obj["Body"].read()), engine="pyarrow")
else:
    parquet_path = Path(LOCAL_PARQUET)
    if not parquet_path.is_absolute():
        root = REPO_ROOT if "REPO_ROOT" in globals() else _repo_root()
        parquet_path = (root / parquet_path).resolve()
    if not parquet_path.exists():
        raise FileNotFoundError(
            f"{parquet_path} not found.\n"
            "From the project root, run:\n"
            "  python scripts/make_sample_parquet.py --output ./out/vehicles_clean.parquet"
        )
    logger.info("Reading local file: %s", parquet_path)
    df_raw = pd.read_parquet(parquet_path, engine="pyarrow")

print(f"Loaded  : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(3)

In [ ]:
# Schema overview
df_raw.dtypes.to_frame(name="dtype").assign(
    nulls=df_raw.isnull().sum(),
    null_pct=(df_raw.isnull().mean() * 100).round(1),
    nunique=df_raw.nunique(),
)

---
## 3. Exploratory Data Analysis

We look at the target (`price`), key predictors, missing-value rates, and relationships between features.

### 3a. Price distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Target: Price", fontsize=13, fontweight="bold")

# Raw (clipped for readability)
df_raw["price"].clip(upper=100_000).plot.hist(bins=80, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_xlabel("Price ($)")
axes[0].set_title("Raw price (clipped at $100k)")

# Log-transformed
np.log1p(df_raw["price"]).plot.hist(bins=80, ax=axes[1], color="darkorange", edgecolor="white")
axes[1].set_xlabel("log(1 + price)")
axes[1].set_title("Log-transformed price")

plt.tight_layout()
plt.show()

print(df_raw["price"].describe().apply(lambda x: f"${x:,.0f}"))

### Key EDA insights (for presentation)

1. **Right-skewed prices** — log transform tightens the distribution.
2. **Vehicle age & odometer** — negative relationship with price.
3. **Categoricals matter** — condition, manufacturer, fuel shift medians.
4. **Person 1 cleaning** — extreme prices removed; ML does not re-filter.

### 3b. Missing value rates

In [ ]:
miss = (df_raw.isnull().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]

if miss.empty:
    print("No missing values found in this dataset.")
else:
    fig, ax = plt.subplots(figsize=(10, max(3, len(miss) * 0.4)))
    miss.plot.barh(ax=ax, color="salmon", edgecolor="white")
    ax.set_xlabel("% Missing")
    ax.set_title("Missing Value Rate by Column")
    for p in ax.patches:
        ax.annotate(
            f"{p.get_width():.1f}%",
            (p.get_width() + 0.3, p.get_y() + 0.2),
            fontsize=8,
        )
    plt.tight_layout()
    plt.show()

### 3c. Numeric correlations

In [ ]:
# Add vehicle_age for correlation view
_tmp = df_raw.copy()
if "year" in _tmp.columns:
    _tmp["vehicle_age"] = datetime.now(timezone.utc).year - _tmp["year"]

num_cols = _tmp.select_dtypes(include="number").columns.tolist()
corr = _tmp[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Numeric Feature Correlation Matrix", fontsize=12)
plt.tight_layout()
plt.show()

### 3d. Price by category

In [ ]:
cat_cols = [c for c in ("manufacturer", "condition", "fuel", "transmission", "type") if c in df_raw.columns]
fig, axes = plt.subplots(len(cat_cols), 1, figsize=(13, 5 * len(cat_cols)))

for ax, col in zip(axes, cat_cols):
    top = df_raw.groupby(col)["price"].median().sort_values(ascending=False).head(12).index
    sub = df_raw[df_raw[col].isin(top)]
    sns.barplot(data=sub, x=col, y="price", order=top, estimator=np.median,
                ax=ax, palette="Blues_d", errorbar=None)
    ax.set_title(f"Median Price by {col.title()}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()

### 3e. Odometer and vehicle age vs price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Odometer scatter (sample for speed)
if "odometer" in df_raw.columns:
    sample = df_raw[["odometer", "price"]].dropna().sample(min(8_000, len(df_raw)), random_state=42)
    axes[0].scatter(sample["odometer"], sample["price"], alpha=0.15, s=5, color="teal")
    axes[0].set_ylim(0, 80_000)
    axes[0].set_xlabel("Odometer (miles)")
    axes[0].set_ylabel("Price ($)")
    axes[0].set_title("Odometer vs Price")

# Vehicle age box plot
if "year" in df_raw.columns:
    _age = df_raw.copy()
    _age["vehicle_age"] = datetime.now(timezone.utc).year - _age["year"]
    _age = _age[(_age["vehicle_age"] >= 0) & (_age["vehicle_age"] <= 40)]
    labels = ["0–3", "4–7", "8–12", "13–20", "21+"]
    _age["age_bucket"] = pd.cut(_age["vehicle_age"], bins=[0, 3, 7, 12, 20, 40], labels=labels)
    sns.boxplot(data=_age[_age["price"] <= 80_000],
                x="age_bucket", y="price", ax=axes[1], color="mediumpurple")
    axes[1].set_title("Price by Vehicle Age Group")
    axes[1].set_xlabel("Age (years)")
    axes[1].set_ylabel("Price ($)")

plt.tight_layout()
plt.show()

---
## 4. Feature Engineering

This section defines the exact supervised-learning problem used by all three models.

- **Target (`y`)**: `price`, the cleaned listing price from Person 1's processed Parquet.
- **Features (`X`)**: the non-target vehicle attributes after lightweight feature engineering and column dropping.
- **Shared preprocessing**: Ridge, Random Forest, and XGBoost all use the same `ColumnTransformer`, so model comparisons are fair. The only thing that changes is the final estimator.

Feature choices are intentionally conservative for the course project: use reliable structured columns, avoid identifiers/leakage, and keep the pipeline reproducible from `config.yaml`.

### 4a. Raw feature construction

In [ ]:
def engineer_features(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """Add derived columns and drop identifiers/leakage columns."""
    df = df.copy()
    feat = cfg["features"]

    # vehicle_age: more interpretable than raw year
    if "year" in df.columns:
        current_year = datetime.now(timezone.utc).year
        df["vehicle_age"] = current_year - df["year"].astype(float)
        print(f"  + vehicle_age  (current_year={current_year})")

    # normalise text categoricals
    cat_cols = list(feat.get("nominal_features", [])) + list(feat.get("ordinal_features", {}).keys())
    for col in cat_cols:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].str.strip().str.lower()

    # drop identifier / leakage columns
    drop_cols = [c for c in feat.get("drop_columns", []) if c in df.columns]
    df = df.drop(columns=drop_cols)
    print(f"  - dropped: {drop_cols}")

    return df


df_eng = engineer_features(df_raw, CFG)
print(f"\nShape after engineering: {df_eng.shape}")
df_eng.head(3)

### 4b. Split features and target

`y` is the value we predict: `price`.

`X` is everything the models are allowed to use for prediction after `engineer_features()` has run. In this notebook, the useful inputs are:

- Numeric: `odometer`, `vehicle_age`
- Ordinal categoricals: `condition`, `cylinders`, `title_status`
- Nominal categoricals: `manufacturer`, `fuel`, `transmission`, `drive`, `type`, `paint_color`, `state`

The same `X_train`, `X_test`, `y_train`, and `y_test` are used for Ridge, Random Forest, and XGBoost.

In [ ]:
TARGET = CFG["features"]["target_column"]   # "price"

# Drop rows with null target
df_eng = df_eng.dropna(subset=[TARGET])

y = df_eng[TARGET].astype(float)
X = df_eng.drop(columns=[TARGET])

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"y range : ${y.min():,.0f} – ${y.max():,.0f}  (median ${y.median():,.0f})")

### 4c. Train / test split

In [ ]:
TEST_SIZE   = CFG["training"]["test_size"]    # 0.20
RANDOM_SEED = CFG["training"]["random_seed"]  # 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

print(f"Train : {len(X_train):,} rows")
print(f"Test  : {len(X_test):,} rows")

### 4d. Build the sklearn ColumnTransformer

The feature selection and transformations are driven by `config.yaml`:

- **Dropped columns**: listing IDs, URLs, image/description text, latitude/longitude, posting date, and raw `year`. These are removed because they are identifiers, free text, unstable deployment fields, or replaced by a cleaner engineered feature.
- **Engineered feature**: `vehicle_age = current_year - year`, which is easier for models to learn than raw model year.
- **Numeric pipeline**: median imputation + `StandardScaler`. Scaling matters most for Ridge because linear models are sensitive to feature scale.
- **Ordinal pipeline**: most-frequent imputation + `OrdinalEncoder` with ordered categories for fields where order has meaning, such as `condition`.
- **Nominal pipeline**: most-frequent imputation + `OneHotEncoder(max_categories=20)`. This avoids exploding dimensionality while still preserving the most common categories.

Can this be better? Yes, but carefully. Next improvements would be: test whether `region` helps, bucket rare manufacturers more intentionally, add interaction features like `vehicle_age * odometer`, and try a log target (`log1p(price)`) if MAPE is the priority. We should only keep additions that improve cross-validation, not just training score.

### 4e. Feature-by-feature explanation

The raw `X` table contains the following candidate predictors after engineering. The model does **not** see these columns exactly as raw strings; the `ColumnTransformer` converts them into a numeric matrix.

**Numeric features**

- `odometer`: mileage in miles. Used as a continuous numeric feature. We impute missing values with the median, then scale it for Ridge. For tree models, scaling is not necessary mathematically, but keeping one shared preprocessor makes the model comparison clean.
- `vehicle_age`: engineered from `year` as `current_year - year`. Used as a continuous numeric feature. This usually captures depreciation better than raw `year` because higher age should generally mean lower price.

**Ordinal categorical features**

- `condition`: encoded with an explicit order from worse to better (`salvage`, `fair`, `good`, `excellent`, `like new`, `new`). This gives Ridge and tree models a numeric signal that better condition should generally increase price.
- `cylinders`: encoded in a rough order by cylinder count (`3`, `4`, `5`, `6`, `8`, `10`, `12`, `other`). This can capture engine size/power. It is not perfectly linear because `other` is mixed, but it is more informative than treating every value as unrelated.
- `title_status`: encoded with an ordered list from problematic to clean (`missing`, `parts only`, `salvage`, `rebuilt`, `lien`, `clean`). This helps represent ownership/title risk.

**Nominal categorical features**

- `manufacturer`: one-hot encoded. We do not impose an order because brands are categories, not ranks.
- `fuel`: one-hot encoded (`gas`, `diesel`, `hybrid`, `electric`, etc.). Fuel type may affect price and demand.
- `transmission`: one-hot encoded because `automatic`, `manual`, and `other` are not naturally ordered.
- `drive`: one-hot encoded (`fwd`, `rwd`, `4wd`) because drivetrain categories are distinct types.
- `type`: one-hot encoded (`sedan`, `suv`, `truck`, etc.) because vehicle body type strongly affects price.
- `paint_color`: one-hot encoded. Color is a weaker feature, but it can capture market preferences or resale effects.
- `state`: one-hot encoded. State can capture regional pricing differences, climate, tax/market differences, and supply/demand.

**Currently present but not selected**

- `region`: this remains in the raw `X` table, but it is **not currently selected** in `config.yaml`, so `ColumnTransformer(remainder="drop")` drops it before modeling. This was conservative because real Craigslist `region` can have higher cardinality and overlap with `state`. To improve the model, we can test adding `region` to `nominal_features` with `max_categories` controlling rare regions. We should keep it only if cross-validation improves.

**How the transformed `X` is used by each model**

- Ridge receives a dense numeric matrix after imputation, scaling, ordinal encoding, and one-hot encoding. It learns one coefficient per transformed feature.
- Random Forest receives the same transformed matrix, then learns many decision trees using feature thresholds and category indicator splits.
- XGBoost receives the same transformed matrix, then builds boosted trees sequentially, where later trees correct earlier errors.

So the three models use the **same input information**, but they learn different relationships from it: Ridge learns mostly additive linear effects; Random Forest and XGBoost can learn non-linear thresholds and interactions.

In [ ]:
# Feature selection audit: what enters the model vs. what is dropped?
feat_cfg = CFG["features"]
present = set(X_train.columns)

audit_numeric_cols = [c for c in feat_cfg.get("numeric_features", []) if c in present]
audit_ordinal_cols = [c for c in feat_cfg.get("ordinal_features", {}).keys() if c in present]
audit_nominal_cols = [c for c in feat_cfg.get("nominal_features", []) if c in present]

configured_features = set(audit_numeric_cols + audit_ordinal_cols + audit_nominal_cols)
raw_x_features = set(X_train.columns)
not_selected = sorted(raw_x_features - configured_features)

feature_audit = pd.DataFrame(
    [
        {"raw_column": col, "role": "numeric", "transformation": "median impute + StandardScaler"}
        for col in audit_numeric_cols
    ]
    + [
        {"raw_column": col, "role": "ordinal", "transformation": "most-frequent impute + OrdinalEncoder"}
        for col in audit_ordinal_cols
    ]
    + [
        {"raw_column": col, "role": "nominal", "transformation": "most-frequent impute + OneHotEncoder"}
        for col in audit_nominal_cols
    ]
    + [
        {"raw_column": col, "role": "not selected", "transformation": "dropped by ColumnTransformer remainder='drop'"}
        for col in not_selected
    ]
)

feature_audit

### 4f. Encoding strategy: what each model actually receives

We are **not using embeddings** in this project. Embeddings are usually used with neural networks when we have very high-cardinality categorical variables and a lot of data. For this course project, a sklearn-style preprocessing pipeline is simpler, easier to explain, and easier to deploy on EC2/S3.

Current shared encoding strategy:

| Feature group | Columns | Encoding / transformation | Why |
|---|---|---|---|
| Numeric | `odometer`, `vehicle_age` | Median imputation + `StandardScaler` | Handles missing numeric values and puts features on comparable scale for Ridge. |
| Ordinal categorical | `condition`, `cylinders`, `title_status` | Most-frequent imputation + `OrdinalEncoder` | These categories have a meaningful order, so a ranked numeric code is reasonable. |
| Nominal categorical | `manufacturer`, `fuel`, `transmission`, `drive`, `type`, `paint_color`, `state` | Most-frequent imputation + `OneHotEncoder(max_categories=20)` | These categories do not have a natural order, so one-hot avoids implying false ranking. |
| Not currently selected | `region` | Dropped by `ColumnTransformer(remainder="drop")` | Conservative choice; can be tested later because region may be higher cardinality and partly redundant with state. |

Important correction about tree models:

1. **Ridge** definitely needs numeric encoded inputs. Each one-hot category gets a coefficient. Scaling is important.
2. **scikit-learn Random Forest** does **not** support raw string categorical variables natively. It still needs numeric inputs. One-hot encoding is safe; ordinal encoding is more compact but can accidentally impose fake ordering on unordered fields.
3. **newer XGBoost can support categorical features natively** if categorical columns are pandas `category` dtype and the model is created with `enable_categorical=True` (usually with `tree_method="hist"`). That means we can compare two XGBoost strategies: one-hot XGBoost and native-categorical XGBoost.

Best practical strategy here:

- Keep the shared one-hot/ordinal pipeline as the main baseline because it works for **all three** models and makes comparisons fair.
- Add a second XGBoost experiment with **native categorical support**. Keep it if cross-validation and test metrics improve.
- For Random Forest, keep one-hot encoding unless dimensionality becomes too large. Native categorical splitting is not available in sklearn's `RandomForestRegressor`.
- Consider target encoding later for high-cardinality fields like `model` or `region`, but only inside cross-validation to avoid target leakage.
- Do **not** use embeddings unless we switch to a neural network model; that would add complexity without helping the cloud-engineering goal much.

So the best current strategy is: **shared sklearn preprocessing for Ridge/RF/XGB + an optional native-categorical XGBoost challenger**.

In [ ]:
feat_cfg = CFG["features"]
present  = set(X_train.columns)

numeric_cols = [c for c in feat_cfg.get("numeric_features", [])  if c in present]
ordinal_cols = [c for c in feat_cfg.get("ordinal_features", {}).keys() if c in present]
nominal_cols = [c for c in feat_cfg.get("nominal_features", [])  if c in present]

print("Numeric columns :", numeric_cols)
print("Ordinal columns :", ordinal_cols)
print("Nominal columns :", nominal_cols)

In [ ]:
# ── Numeric pipeline ───────────────────────────────────────────────────────
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

# ── Ordinal pipeline ───────────────────────────────────────────────────────
ordinal_categories = [
    [str(c) for c in feat_cfg["ordinal_features"][col]["categories"]]
    for col in ordinal_cols
]

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=ordinal_categories if ordinal_categories else "auto",
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
])

# ── Nominal pipeline ───────────────────────────────────────────────────────
nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        max_categories=feat_cfg.get("ohe_max_categories", 20),
        handle_unknown="infrequent_if_exist",
        sparse_output=False,
    )),
])

# ── Assemble ───────────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric",  numeric_pipe,  numeric_cols),
        ("ordinal",  ordinal_pipe,  ordinal_cols),
        ("nominal",  nominal_pipe,  nominal_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Quick sanity-check: fit on train, transform a tiny sample
_X_sample = preprocessor.fit_transform(X_train)
print(f"Preprocessor output shape: {_X_sample.shape}")
print(f"Features produced        : {_X_sample.shape[1]}")

In [ ]:
# Inspect the feature names the transformer produces
feature_names = list(preprocessor.get_feature_names_out())
print(f"Total output features: {len(feature_names)}")
print("First 20:", feature_names[:20])

---
## 5. Linear Baseline — Ridge

We use **Ridge regression** as the linear baseline instead of plain Linear Regression because the preprocessing creates many one-hot encoded columns that can be correlated. Ridge adds a small L2 penalty (`alpha`) that stabilizes coefficients and reduces overfitting while keeping the model fast and interpretable.

This is our Week 7 baseline story: if a simple regularized linear model performs reasonably, then Random Forest and XGBoost need to justify their extra complexity by improving validation/test metrics.

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1, None))) * 100
    print(f"  {name:12s}  RMSE ${rmse:>9,.0f}   MAE ${mae:>8,.0f}   R² {r2:.4f}   MAPE {mape:.1f}%")
    return {"rmse": rmse, "mae": mae, "r2": r2, "mape_pct": mape}

cv_n_jobs = CFG["training"].get("cv_n_jobs", 1)

In [ ]:
ridge_params = CFG["training"]["ridge"]
pre_ridge = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe, numeric_cols),
        ("ordinal", ordinal_pipe, ordinal_cols),
        ("nominal", nominal_pipe, nominal_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
ridge_pipeline = Pipeline([
    ("preprocessor", pre_ridge),
    ("model", Ridge(**ridge_params)),
])

cv_ridge = cross_val_score(
    ridge_pipeline, X_train, y_train,
    cv=CFG["training"]["cv_folds"], scoring="r2", n_jobs=cv_n_jobs,
)
ridge_pipeline.fit(X_train, y_train)
ridge_train_metrics = evaluate("Ridge train", y_train.values, ridge_pipeline.predict(X_train))
ridge_test_metrics = evaluate("Ridge test", y_test.values, ridge_pipeline.predict(X_test))
ridge_cv_r2 = cv_ridge.mean()
print(f"CV R²: {ridge_cv_r2:.4f}")

---
## 6. Tree Baseline — Random Forest

Random Forest is the stronger non-linear baseline. It can learn threshold effects such as “newer cars with low odometer readings are worth more” without us manually writing every interaction.

The starting hyperparameters in `config.yaml` are conservative:

- `n_estimators=200`: enough trees for stable performance without making EC2 training too slow.
- `max_depth=20`: limits overly deep trees.
- `min_samples_leaf=4`: smooths predictions by requiring each leaf to represent multiple listings.

We do not call these “best” yet. They are reasonable defaults; the tuning section below shows how to choose better values with cross-validation.

In [ ]:
rf_params = CFG["training"]["random_forest"]
print("Random Forest hyperparameters (from config.yaml):")
for k, v in rf_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Rebuild a fresh preprocessor (not yet fitted) for the pipeline
pre_rf = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe,  numeric_cols),
        ("ordinal", ordinal_pipe,  ordinal_cols),
        ("nominal", nominal_pipe,  nominal_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

rf_pipeline = Pipeline([
    ("preprocessor", pre_rf),
    ("model", RandomForestRegressor(**rf_params)),
])

# Cross-validation on training set
print("Running 5-fold CV on training set...")
t0 = time.time()
cv_rf = cross_val_score(rf_pipeline, X_train, y_train, cv=CFG["training"]["cv_folds"],
                        scoring="r2", n_jobs=cv_n_jobs)
print(f"  CV R²: {cv_rf.mean():.4f} ± {cv_rf.std():.4f}  ({time.time()-t0:.1f}s)")

# Final fit
print("Fitting on full training set...")
t0 = time.time()
rf_pipeline.fit(X_train, y_train)
print(f"  Fit done in {time.time()-t0:.1f}s")

# Evaluate
print("\nMetrics:")
rf_train_metrics = evaluate("RF train",  y_train.values, rf_pipeline.predict(X_train))
rf_test_metrics  = evaluate("RF test",   y_test.values,  rf_pipeline.predict(X_test))
rf_cv_r2 = cv_rf.mean()

---
## 7. Challenger Model — XGBoost

XGBoost is the challenger model because gradient boosting often performs well on structured tabular data. Unlike Random Forest, it builds trees sequentially, where each new tree focuses on errors from earlier trees.

The main hyperparameters control different tradeoffs:

- `n_estimators` and `learning_rate`: more trees with a smaller learning rate usually generalizes better but trains slower.
- `max_depth`: deeper trees learn stronger interactions but overfit more easily.
- `subsample` and `colsample_bytree`: add randomness to reduce overfitting.
- `reg_alpha` and `reg_lambda`: L1/L2 regularization.
- `early_stopping_rounds`: in the script version, a validation split stops boosting when validation performance stops improving.

As with Random Forest, the current values are a good first pass. Cross-validation or randomized search is how we choose better values.

### 7b. XGBoost with native categorical support

Newer XGBoost versions can handle categorical predictors directly. For this path, we do **not** one-hot encode nominal features. Instead:

- numeric columns stay numeric: `odometer`, `vehicle_age`
- categorical columns are converted to pandas `category` dtype
- `XGBRegressor(enable_categorical=True, tree_method="hist")` lets XGBoost decide category splits internally

This is **not available for sklearn Random Forest**, but it is worth testing for XGBoost. We keep the one-hot XGBoost above as the stable baseline, then compare this native-categorical version below.

In [ ]:
# Native-categorical XGBoost experiment.
# This path uses a different X matrix from Ridge/RF: raw categoricals are kept as pandas category dtype.

xgb_native_feature_cols = numeric_cols + ordinal_cols + nominal_cols
# Optional experiment: include region as native categorical if present.
if "region" in X_train.columns and "region" not in xgb_native_feature_cols:
    xgb_native_feature_cols = xgb_native_feature_cols + ["region"]

xgb_native_cat_cols = [c for c in ordinal_cols + nominal_cols + ["region"] if c in xgb_native_feature_cols]
xgb_native_num_cols = [c for c in numeric_cols if c in xgb_native_feature_cols]


def prepare_xgb_native_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame[xgb_native_feature_cols].copy()
    for col in xgb_native_num_cols:
        out[col] = out[col].fillna(out[col].median()).astype(float)
    for col in xgb_native_cat_cols:
        out[col] = out[col].astype("string").fillna("missing").astype("category")
    return out


X_train_xgb_native = prepare_xgb_native_frame(X_train)
X_test_xgb_native = prepare_xgb_native_frame(X_test)

xgb_native_params = {
    k: v
    for k, v in CFG["training"]["xgboost"].items()
    if k not in ("early_stopping_rounds", "validation_size")
}
xgb_native_params.update({"enable_categorical": True, "tree_method": "hist"})

xgb_native = XGBRegressor(**xgb_native_params)

print("Native categorical XGBoost features:")
print("Numeric     :", xgb_native_num_cols)
print("Categorical :", xgb_native_cat_cols)

print("Running 5-fold CV on training set...")
t0 = time.time()
cv_xgb_native = cross_val_score(
    xgb_native,
    X_train_xgb_native,
    y_train,
    cv=CFG["training"]["cv_folds"],
    scoring="r2",
    n_jobs=cv_n_jobs,
)
print(f"  CV R²: {cv_xgb_native.mean():.4f} ± {cv_xgb_native.std():.4f}  ({time.time()-t0:.1f}s)")

print("Fitting native-categorical XGBoost...")
t0 = time.time()
xgb_native.fit(X_train_xgb_native, y_train)
print(f"  Fit done in {time.time()-t0:.1f}s")

print("\nMetrics:")
xgb_native_train_metrics = evaluate(
    "XGB-native train", y_train.values, xgb_native.predict(X_train_xgb_native)
)
xgb_native_test_metrics = evaluate(
    "XGB-native test", y_test.values, xgb_native.predict(X_test_xgb_native)
)
xgb_native_cv_r2 = cv_xgb_native.mean()

In [ ]:
from xgboost import XGBRegressor

# Keep notebook-only metadata out of the estimator constructor.
xgb_params = {
    k: v
    for k, v in CFG["training"]["xgboost"].items()
    if k not in ("early_stopping_rounds", "validation_size")
}
print("XGBoost hyperparameters (from config.yaml):")
for k, v in xgb_params.items():
    print(f"  {k}: {v}")

In [ ]:
pre_xgb = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipe,  numeric_cols),
        ("ordinal", ordinal_pipe,  ordinal_cols),
        ("nominal", nominal_pipe,  nominal_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

xgb_pipeline = Pipeline([
    ("preprocessor", pre_xgb),
    ("model", XGBRegressor(**xgb_params)),
])

# Cross-validation
print("Running 5-fold CV on training set...")
t0 = time.time()
cv_xgb = cross_val_score(xgb_pipeline, X_train, y_train, cv=CFG["training"]["cv_folds"],
                         scoring="r2", n_jobs=cv_n_jobs)
print(f"  CV R²: {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}  ({time.time()-t0:.1f}s)")

# Final fit
print("Fitting on full training set...")
t0 = time.time()
xgb_pipeline.fit(X_train, y_train)
print(f"  Fit done in {time.time()-t0:.1f}s")

# Evaluate
print("\nMetrics:")
xgb_train_metrics = evaluate("XGB train", y_train.values, xgb_pipeline.predict(X_train))
xgb_test_metrics  = evaluate("XGB test",  y_test.values,  xgb_pipeline.predict(X_test))
xgb_cv_r2 = cv_xgb.mean()

---
## 8. Optional Hyperparameter Tuning

The models above use sensible defaults from `config.yaml`. To make the modeling story stronger, we should choose hyperparameters using **cross-validation on the training set only**, then evaluate the selected model once on the held-out test set.

For this project, a small `RandomizedSearchCV` is a good fit:

- It is cheaper than a full grid search on EC2.
- It explores the most important parameters.
- It keeps the held-out test set clean for final reporting.

Important: leave `RUN_TUNING = False` during quick notebook demos. Turn it on only when you are ready to spend a few minutes searching.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

RUN_TUNING = False  # Set True for an EC2 run or when you want to update config.yaml.
N_ITER = 8          # Keep small for class/demo speed; increase on EC2.


def make_preprocessor() -> ColumnTransformer:
    """Fresh unfitted preprocessor for each search/model."""
    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, numeric_cols),
            ("ordinal", ordinal_pipe, ordinal_cols),
            ("nominal", nominal_pipe, nominal_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


rf_search_space = {
    "model__n_estimators": [100, 200, 400],
    "model__max_depth": [10, 15, 20, None],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", 0.5, 0.8, 1.0],
}

xgb_search_space = {
    "model__n_estimators": [200, 400, 600],
    "model__learning_rate": [0.03, 0.05, 0.08, 0.1],
    "model__max_depth": [3, 5, 7],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
    "model__reg_alpha": [0.0, 0.1, 0.5],
    "model__reg_lambda": [0.5, 1.0, 2.0],
}

if RUN_TUNING:
    searches = {}

    rf_search = RandomizedSearchCV(
        estimator=Pipeline([
            ("preprocessor", make_preprocessor()),
            ("model", RandomForestRegressor(n_jobs=-1, random_state=RANDOM_SEED)),
        ]),
        param_distributions=rf_search_space,
        n_iter=N_ITER,
        scoring="r2",
        cv=CFG["training"]["cv_folds"],
        random_state=RANDOM_SEED,
        n_jobs=cv_n_jobs,
        verbose=1,
    )
    rf_search.fit(X_train, y_train)
    searches["RandomForest"] = rf_search

    xgb_search = RandomizedSearchCV(
        estimator=Pipeline([
            ("preprocessor", make_preprocessor()),
            ("model", XGBRegressor(n_jobs=-1, random_state=RANDOM_SEED)),
        ]),
        param_distributions=xgb_search_space,
        n_iter=N_ITER,
        scoring="r2",
        cv=CFG["training"]["cv_folds"],
        random_state=RANDOM_SEED,
        n_jobs=cv_n_jobs,
        verbose=1,
    )
    xgb_search.fit(X_train, y_train)
    searches["XGBoost"] = xgb_search

    for name, search in searches.items():
        print(f"\n{name} best CV R²: {search.best_score_:.4f}")
        print("Best parameters:")
        for param, value in search.best_params_.items():
            print(f"  {param}: {value}")

    # Optional: replace rf_pipeline/xgb_pipeline with tuned versions after reviewing results.
else:
    print("Tuning skipped. Set RUN_TUNING = True to run RandomizedSearchCV.")

---
## 8. Model Comparison

### 8a. Side-by-side metric table

In [ ]:
comparison_rows = [
    {"Model": "Ridge", "CV R²": round(ridge_cv_r2, 4),
     "Test R²": round(ridge_test_metrics["r2"], 4),
     "Test RMSE ($)": round(ridge_test_metrics["rmse"]),
     "Test MAE ($)": round(ridge_test_metrics["mae"]),
     "Test MAPE (%)": round(ridge_test_metrics["mape_pct"], 1)},
    {"Model": "Random Forest", "CV R²": round(rf_cv_r2, 4),
     "Test R²": round(rf_test_metrics["r2"], 4),
     "Test RMSE ($)": round(rf_test_metrics["rmse"]),
     "Test MAE ($)": round(rf_test_metrics["mae"]),
     "Test MAPE (%)": round(rf_test_metrics["mape_pct"], 1)},
    {"Model": "XGBoost (one-hot)", "CV R²": round(xgb_cv_r2, 4),
     "Test R²": round(xgb_test_metrics["r2"], 4),
     "Test RMSE ($)": round(xgb_test_metrics["rmse"]),
     "Test MAE ($)": round(xgb_test_metrics["mae"]),
     "Test MAPE (%)": round(xgb_test_metrics["mape_pct"], 1)},
]

if "xgb_native_test_metrics" in globals():
    comparison_rows.append(
        {"Model": "XGBoost (native categorical)", "CV R²": round(xgb_native_cv_r2, 4),
         "Test R²": round(xgb_native_test_metrics["r2"], 4),
         "Test RMSE ($)": round(xgb_native_test_metrics["rmse"]),
         "Test MAE ($)": round(xgb_native_test_metrics["mae"]),
         "Test MAPE (%)": round(xgb_native_test_metrics["mape_pct"], 1)}
    )

comparison = pd.DataFrame(comparison_rows).set_index("Model")

comparison.style.highlight_max(subset=["CV R²", "Test R²"], color="#d4edda") \
                .highlight_min(subset=["Test RMSE ($)", "Test MAE ($)", "Test MAPE (%)"], color="#d4edda")

In [ ]:
# Which model wins?
_candidates = {
    "Ridge": (ridge_test_metrics, ridge_pipeline),
    "Random Forest": (rf_test_metrics, rf_pipeline),
    "XGBoost (one-hot)": (xgb_test_metrics, xgb_pipeline),
}
if "xgb_native_test_metrics" in globals():
    _candidates["XGBoost (native categorical)"] = (xgb_native_test_metrics, xgb_native)
best_name = max(_candidates, key=lambda k: _candidates[k][0]["r2"])
best_metrics, best_pipeline = _candidates[best_name]
print(f"Best model: {best_name}  (Test R² = {best_metrics['r2']:.4f})")

### 8b. Actual vs predicted plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name, pipe in [(axes[0], "Random Forest", rf_pipeline),
                       (axes[1], "XGBoost",       xgb_pipeline)]:
    y_pred = pipe.predict(X_test)
    lim = min(y_test.max(), 150_000)
    ax.scatter(y_test, y_pred, alpha=0.15, s=6, color="steelblue")
    ax.plot([0, lim], [0, lim], "r--", lw=1.2, label="Perfect prediction")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_xlabel("Actual Price ($)")
    ax.set_ylabel("Predicted Price ($)")
    ax.set_title(f"{name} — Actual vs Predicted")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### 8c. Residual distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, name, pipe in [(axes[0], "Random Forest", rf_pipeline),
                       (axes[1], "XGBoost",       xgb_pipeline)]:
    residuals = y_test.values - pipe.predict(X_test)
    ax.hist(residuals.clip(-50_000, 50_000), bins=80, color="mediumpurple", edgecolor="white")
    ax.axvline(0, color="red", lw=1.5, linestyle="--")
    ax.set_xlabel("Residual ($)")
    ax.set_title(f"{name} — Residual Distribution")

plt.tight_layout()
plt.show()

### 8d. Feature importance (best model)

In [ ]:
feat_names = list(best_pipeline.named_steps["preprocessor"].get_feature_names_out())
model_step = best_pipeline.named_steps["model"]
importances = (
    model_step.feature_importances_
    if hasattr(model_step, "feature_importances_")
    else np.abs(model_step.coef_)
)

imp_df = (
    pd.DataFrame({"feature": feat_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(25)
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=imp_df, y="feature", x="importance", ax=ax, palette="viridis")
ax.set_title(f"Top 25 Feature Importances — {best_name}", fontsize=12)
ax.set_xlabel("Importance")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

---
## 9. Save Artifacts

Prefer `python -m modeling.train` on EC2 (see `infra/ec2_train.sh`). This cell mirrors uploads to `artifacts/models/latest/`.

In [ ]:
def upload_to_s3(bucket, key, data: bytes, content_type: str):
    """Upload bytes to S3 using the attached IAM role (no credentials needed)."""
    import boto3
    boto3.client("s3", region_name=REGION).put_object(
        Bucket=bucket, Key=key, Body=data, ContentType=content_type
    )
    print(f"  ✓ s3://{bucket}/{key}  ({len(data):,} bytes)")


def pickle_bytes(obj) -> bytes:
    buf = io.BytesIO()
    pickle.dump(obj, buf)
    return buf.getvalue()


artifact_cfg = CFG["artifacts"]

metrics_payload = {
    "models": [
        {"model": "Ridge", "cv_r2": ridge_cv_r2, "test": ridge_test_metrics},
        {"model": "RandomForest", "cv_r2": rf_cv_r2, "test": rf_test_metrics},
        {"model": "XGBoost", "cv_r2": xgb_cv_r2, "test": xgb_test_metrics},
    ],
    "best_model": best_name,
}
manifest = {
    "best_model": best_name,
    "target_column": CFG["features"]["target_column"],
    "use_log_target": CFG["training"].get("use_log_target", False),
}

if USE_S3:
    print("Uploading artifacts to S3...")
    upload_to_s3(BUCKET, artifact_cfg["ridge_model_key"], pickle_bytes(ridge_pipeline), "application/octet-stream")
    upload_to_s3(BUCKET, artifact_cfg["baseline_model_key"], pickle_bytes(rf_pipeline), "application/octet-stream")
    upload_to_s3(BUCKET, artifact_cfg["challenger_model_key"], pickle_bytes(xgb_pipeline), "application/octet-stream")
    upload_to_s3(BUCKET, artifact_cfg["best_model_key"], pickle_bytes(best_pipeline), "application/octet-stream")
    upload_to_s3(BUCKET, artifact_cfg["metrics_key"], json.dumps(metrics_payload, indent=2).encode(), "application/json")
    upload_to_s3(BUCKET, artifact_cfg["manifest_key"], json.dumps(manifest, indent=2).encode(), "application/json")
    upload_to_s3(BUCKET, artifact_cfg["feature_importance_key"], imp_df.to_csv(index=False).encode(), "text/csv")
    print("\nAll artifacts uploaded to artifacts/models/latest/")
else:
    out = Path(LOCAL_ARTIFACTS)
    out.mkdir(parents=True, exist_ok=True)
    for name, pipe in [("ridge.pkl", ridge_pipeline), ("random_forest.pkl", rf_pipeline),
                       ("xgboost.pkl", xgb_pipeline), ("best_model.pkl", best_pipeline)]:
        with open(out / name, "wb") as f:
            pickle.dump(pipe, f)
    (out / "metrics.json").write_text(json.dumps(metrics_payload, indent=2))
    (out / "model_manifest.json").write_text(json.dumps(manifest, indent=2))
    imp_df.to_csv(out / "feature_importance.csv", index=False)
    print(f"Artifacts saved to {out.resolve()}")

---
## Summary

| Step | Output |
|---|---|
| EDA | Price distributions, missing values, correlations, category plots |
| Feature Engineering | `vehicle_age` derived from `year`; ordinal + OHE encoding; median/mode imputation |
| Linear baseline (Ridge) | `artifacts/models/latest/ridge.pkl` |
| Tree baseline (Random Forest) | `artifacts/models/latest/random_forest.pkl` |
| Challenger (XGBoost) | `artifacts/models/latest/xgboost.pkl` |
| Best model (Person 3) | `artifacts/models/latest/best_model.pkl` + `model_manifest.json` |
| Metrics | `artifacts/models/latest/metrics.json` |

All hyperparameters are sourced from `config.yaml` — nothing is hardcoded.  
For production training, run `python -m modeling.train --config config.yaml` on EC2 using `infra/ec2_train.sh`.